Production mein is architecture ko kaise improve karoge?

PRODUCTION-ORIENTED

Document ingestion
 ↓
Parsing/OCR
 ↓
Structure-aware chunking
 ↓
Embedding model
 ↓
Persistent Vector DB
 ↓
Metadata filtering
 ↓
Retriever
 ↓
Reranker
 ↓
Context compression
 ↓
RAG Tool
 ↓
LangGraph Agent
 ↓
LLM
 ↓
Citations / grounded answer

Aur production mein kuch additional concerns bhi honge:
- Persistent vector database instead of an in-memory FAISS index.
- Async ingestion/retrieval where appropriate.
- Document IDs + metadata for source tracking.
- Authentication/authorization if different users have different documents.
- Prompt-injection protection for untrusted documents.
- Retrieval evaluation — sirf "answer acha lag raha hai" enough nahi.
- Tracing/observability with something like LangSmith.
- Caching where useful.
- Reranking when basic similarity retrieval isn't enough.
- Chunking evaluation rather than assuming 1000/200.
- Citations so the user can see where the answer came from.
- Error handling/timeouts/retries for model and embedding APIs.
- Secrets in environment/secret manager, never hard-coded API keys.

Tumhare liye sabse important learning
Is poore code mein 5 cheezen deeply samjho:
1. Chunking
       ↓
2. Embeddings
       ↓
3. Vector Store
       ↓
4. Retriever
       ↓
5. RAG as a LangGraph Tool

Aur phir:
LLM
 ↓
Tool decision
 ↓
ToolNode
 ↓
RAG
 ↓
LLM

Ye last wala part tumhari Agentic AI learning ke liye particularly important hai, because ab tum simple "RAG chain" nahi bana rahe — tum RAG ko agent ke ek tool ke taur par expose kar rahe ho.

In [45]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from dotenv import load_dotenv

In [46]:
load_dotenv()

llm = ChatGroq(model='openai/gpt-oss-120b')

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1485.63it/s]


In [47]:
#rag pipeline
#loader
loader= PyPDFLoader('pdf_rag_langgraph.pdf')
docs=loader.load()
len(docs)


#chunking
splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks=splitter.split_documents(docs)
len(chunks)

#vector_database
vector_store= FAISS.from_documents(chunks, embedding=embeddings )

#retriver
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [48]:
#RAG become tool now 
@tool
def rag_tool(query):
    """Retrieve relevant information from the pdf document.
Use this tool when the user asks factual / conceptual questions
that might be answered from the stored documents."""

    result= retriever.invoke(query)

    context= [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query':query,
        'context':context,
        'metadata':metadata
    }
    

In [49]:
tools= [rag_tool]
tool_with_llm= llm.bind_tools(tools)

In [50]:
class rag_state(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [51]:
def chat_node(state:rag_state)->rag_state:
    message=state['messages']
    response=tool_with_llm.invoke(message)
    return {
        'messages':[response]
    }

tool_node=ToolNode(tools)


In [58]:
graph = StateGraph(rag_state)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()

In [59]:
query = "Using the PDF, explain KNN"
resultt = chatbot.invoke({
    "messages": [
        HumanMessage(
            content=query
        )
    ]
})

In [60]:
print(resultt["messages"][-1].content)

**k‑Nearest Neighbour (k‑NN) – a concise explanation drawn from the PDF**

---

### 1. What k‑NN does  

| Task | How k‑NN works |
|------|----------------|
| **Classification** | For a new observation we locate the *k* training points that are closest to it (according to a distance metric such as Euclidean). The new point is assigned to the class that appears most often among those *k* neighbours (majority‑vote rule). |
| **Prediction of a continuous value (regression)** | The same neighbourhood is found, but instead of a vote we take the **average** of the target values of the *k* neighbours. Often a **weighted average** is used, giving larger weight to nearer neighbours. |

---

### 2. Why the choice of *k* matters  

* **Small *k* (e.g., *k* = 1)** – very little smoothing. The model can fit the training data almost perfectly, but it is highly sensitive to noise (over‑fitting).  
* **Large *k*** – strong smoothing. The decision surface becomes smoother, reducing variance but risking